# Importación de librerías

In [ ]:
# Módulo
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import new_data_popularity, load_env_file
load_env_file()

# Carga de datos

In [ ]:
use_minio = False
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(new_data_popularity, minio)
df.head(2)

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=df['recomendaciones_totales'],
        marker_color="#409ada",
        name='Recomendaciones',
        hovertemplate='<b>Recomendaciones:</b> %{x}<br><b>Frecuencia:</b> %{y}<extra></extra>',
        xbins=dict(size=1)
    )
)

fig.update_layout(
    title={
        'text': 'Distribución de Recomendaciones Totales en Nuevos Datos',
        'x': 0.5,
        'font': dict(size=18)
    },
    xaxis_title_text='Número de Recomendaciones Totales',
    yaxis_title_text='Frecuencia (Cantidad)',
    plot_bgcolor='whitesmoke'
)

fig.show()

In [ ]:
display(df.sort_values(by='recomendaciones_totales', ascending=False).head(20).reset_index(drop=True))

In [ ]:
df_cleaned = df.copy()
df_cleaned["viewCountTotal"] = df_cleaned.filter(like="video_statistics.viewCount").sum(axis=1)
df_cleaned["likeCountTotal"] = df_cleaned.filter(like="video_statistics.likeCount").sum(axis=1)
df_cleaned["commentCountTotal"] = df_cleaned.filter(like="video_statistics.commentCount").sum(axis=1)

numeric_columns = ["recomendaciones_totales", "description_len", "price_overview", "num_languages", "num_juegos_previos_developers", "ema_reviews_developers", "max_historico_reviews_developers", \
            "num_juegos_previos_publishers", "ema_reviews_publishers", "max_historico_reviews_publishers", "brillo", "viewCountTotal", \
            "likeCountTotal", "commentCountTotal"]

corr = df_cleaned[numeric_columns].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', annot=True)
plt.show()

In [ ]:
columnas_mayor_correlacion = ["price_overview","num_juegos_previos_developers", "num_juegos_previos_publishers"]
for col in columnas_mayor_correlacion:
    fig = px.histogram(df_cleaned, x=col, title=f"Distribucion de {col}")
    fig.show()